# Analytical null integration — summary

**Task** `analytical-null-md`. This notebook is the review artifact
for the pull request: read this first.

## What this task did

The web query tool's null (`query_metapath_z` in `src/multi_dwpc_query.py`)
is replaced. The old null scored a user's gene set against `b = 20` random
same-size gene subsets drawn from the whole gene universe. The new null is the exact moments of the
same resampling scheme, computed in closed form by
`hetnetex_md.exact_resampling_moments` over a **capacity-stratified**
partition of the gene universe, via the adapter `analytical_gene_set_z`
(`src/analytical_null.py`).

## The hypotheses

`design.md`'s "Expected result" section (and its "Behaviour changes" claim
1, Deterministic) stated pre-run expectations, approved before this task's
verify step ran a single query. 

1. "The example query's analytical z correlates with the Monte-Carlo z."
2. "Seed-to-seed MC spread is visible at default `b`; the analytical value
   has none."
3. "Query latency drops by an order of magnitude or more."


## Why stratify

The null must answer: is this gene set's connectivity to *this* target
higher than a random gene set's would be? That depends on what counts as "a
random gene set." A gene's raw DWPC to a target is confounded by its overall
connectivity (degree) in the network — high-degree genes score higher
against every target, enrichment or not. Stratifying removes that confound:
each of the user's genes is compared only against genes of similar
**capacity** (its total raw DWPC to every *other* target of the metapath,
excluding the tested target), so the resulting z-score reports whether these
genes reach this target more than genes with the same overall reach — not
more than an unstratified random draw would.

## What to expect below

- **Adapt-step evidence** 
- **Verify-step evidence** 
- **Behavior changes** 
- **Conclusions** 


## Adapt-step evidence

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

# Robust to being run either from this notebook's own directory (the
# intended way — see plan.md/design.md) or from the repository root.
BASE = Path.cwd()
if not (BASE / "tables").exists():
    BASE = Path("docs/tasks/analytical-null-md")
if not (BASE / "tables").exists():
    raise FileNotFoundError(
        "Could not locate tables/ and figures/. Run this notebook from "
        "docs/tasks/analytical-null-md/, its own directory."
    )

TABLES = BASE / "tables"
FIGURES = BASE / "figures"
print(f"Reading tables from:  {TABLES.resolve()}")
print(f"Reading figures from: {FIGURES.resolve()}")


In [ ]:
comparison = pd.read_csv(TABLES / "per_metapath_comparison.csv")
display(comparison.head())

n_total = len(comparison)
n_nan_analytical = comparison["z_analytical"].isna().sum()
n_nan_any_mc_seed = (
    comparison[["z_mc_seed42", "z_mc_seed43", "z_mc_seed44"]].isna().any(axis=1).sum()
)

print(f"Metapaths scored: {n_total}")
print(f"Analytical z is NaN (zero-variance null): {n_nan_analytical} / {n_total}")
print(f"NaN in at least one MC seed:              {n_nan_any_mc_seed} / {n_total}")


In [ ]:
Image(filename=str(FIGURES / "old_vs_new_z_scatter.png"))


**Figure 1 — old vs. new z, per metapath.** The Monte-Carlo z at each of
three seeds (42, 43, 44) plotted against the single analytical z, with
`y = x` for reference. The MC seeds sit systematically above the `y = x`
line for the highest-z metapaths — the naive, degree-blind null over-states
enrichment there relative to the degree-stratified analytical null — and
scatter on both sides of it near `z = 0`. Read plainly: the analytical z
sits generally at or below the Monte-Carlo z.


In [ ]:
Image(filename=str(FIGURES / "mc_seed_spread.png"))


**Figure 2 — Monte-Carlo seed spread vs. the analytical value.**
Per-metapath MC seed range (min-max across seeds 42/43/44, horizontal bars)
against the single deterministic analytical z (dot) — the determinism claim
made visible. Several metapaths (`GpPWpGpBP`, `GpBPpGpBP`, `GpMFpGpBP`) show
MC seed ranges spanning roughly 5-20 z units (min-max: 19.34, 15.88, 4.75
respectively) at `b = 20`; the analytical value has none, by construction —
the same query returns the same z every time.


## Verify-step evidence: timing

In [ ]:
timing = pd.read_csv(TABLES / "timing.csv")
warm_timing = pd.read_csv(TABLES / "warm_metapath_timing.csv")

display(timing)
display(warm_timing)

print("End-to-end median wall-clock (s), 3 runs each:")
display(timing.groupby("implementation")["wall_clock_s"].median())

print("Warm-matrix median wall-clock (s), 5 runs each:")
display(warm_timing.groupby(["metapath", "implementation"])["wall_clock_s"].median())


In [ ]:
Image(filename=str(FIGURES / "timing_comparison.png"))


**Figure 3 — timing comparison.** Two panels, one per table above.

**Hypothesis** (`design.md`, Expected result, as approved): "Query latency
drops by an order of magnitude or more."

**Measured:** end-to-end (52 metapaths, disk-bound, median of 3 runs each)
— new (analytical) 103 s vs old (Monte-Carlo, `b = 20`) 85 s. Warm-matrix
(matrix already resident, isolates the null-computation cost alone, median
of 5 runs each) — 70 ms vs 8.6 ms on `GiGiGpBP`, 12.6 ms vs 7.5 ms on the
smaller `GpBP`.

**Verdict: hypothesis NOT met.** The analytical implementation is slower,
not faster, at both grains measured, and the gap runs the opposite
direction from an order-of-magnitude drop.

**Interpretation** (post-hoc, not predicted by the design; offered as a
candidate explanation, not a second measurement):

- Both implementations spend most of end-to-end wall-clock on disk I/O
  loading DWPC matrices, not on computing the null, so the end-to-end
  number alone does not isolate the algorithmic cost; the warm-matrix
  figures do, and they still show the analytical implementation slower.
- At the app's default `b = 20`, the old null was fast because it was
  imprecise: a null standard deviation estimated from only 20 draws carries
  roughly 16% relative error, which every z-score it feeds inherits. Twenty
  draws is cheap; twenty draws is also not a precise estimate.
- The analytical moments price out at the Monte-Carlo sample size a
  comparable-precision null would need — validated at 214x against
  `B = 1,000` draws and 2,145x against `B = 10,000` draws (the
  capacity-hurdle-adaptive validation task, see Conclusions), not against
  `b = 20`. At the app's much smaller `b = 20`, that crossover has not been
  reached for the metapaths measured here.

This interpretation explains why the hypothesis failed at this scale; it
does not change the verdict above.


## Verify-step evidence: rank agreement

In [ ]:
rank_agreement = pd.read_csv(TABLES / "rank_agreement.csv")
display(rank_agreement.head())
print(f"Rows with both an analytical and an MC-seed-42 rank (n): {len(rank_agreement)}")


In [ ]:
Image(filename=str(FIGURES / "rank_agreement.png"))


**Figure 4 — rank agreement.** Analytical rank vs. Monte-Carlo (seed 42)
rank for the `n = 44` metapaths where both are defined (52 total minus 8
where either side is a zero-variance NaN).

**Hypothesis** (`design.md`, Expected result, as approved): "The example
query's analytical z correlates with the Monte-Carlo z."

**Measured:** Spearman rho = 0.2605 (p = 0.0877, n = 44), reported in
`verification.md` — weaker than the hypothesis implied, and not significant
at alpha = 0.05.

**Verdict: hypothesis only weakly supported.** The correlation is positive
but not the strong agreement "correlates with" would ordinarily be read to
promise; the top-ranked metapaths (`GpBPpGpBP`, `GpPWpGpBP`, `GpMFpGpBP`)
agree on both sides, but the middle of the ranking reorders substantially
(the broad scatter around, not tight against, the `y = x` line in the
figure).

**Interpretation** (post-hoc, not predicted by the design; a candidate for
follow-on work to test directly, not itself validated by this run): a null
that is blind to degree (Monte-Carlo, whole-gene-universe) and a null that
is not (analytical, capacity-stratified) would, on the degree-confounding
mechanism `design.md` states, be expected to reorder exactly the metapaths
where degree and enrichment diverge — consistent with what the figure
shows, but this run tested only the aggregate correlation, not that
mechanism directly. As background rather than as evidence this task
produced: the validation task behind this null design saw its own
calibration pass rate fall from 85% to 47% to 14.7% across successive
redesigns as its test was made genuinely discriminating, on the way to the
capacity-hurdle-adaptive (S1) design that ships here — measured in
`docs/tasks/capacity-hurdle-adaptive-null/verification.md` on branch
`fix/random-null-stratified-srswor` (see Conclusions).


## Verify-step evidence: agreement and cost versus B

In [ ]:
b_sweep_summary = pd.read_csv(TABLES / "b_sweep_summary.csv")
display(b_sweep_summary)

wall_time_b10000 = b_sweep_summary.loc[
    (b_sweep_summary["row_type"] == "same_null_mc") & (b_sweep_summary["B"] == 10000),
    "total_wall_time_s",
].iloc[0]
wall_time_analytical = b_sweep_summary.loc[
    b_sweep_summary["row_type"] == "analytical", "total_wall_time_s"
].iloc[0]
ratio = wall_time_b10000 / wall_time_analytical

print(f"Same-null MC, B=10,000, 3-seed total wall-time: {wall_time_b10000:.2f} s")
print(f"Analytical closed form, 1-query wall-time:       {wall_time_analytical:.4f} s")
print(f"Ratio (B=10,000 total wall-time / analytical wall-time): {ratio:,.0f}x")


In [ ]:
Image(filename=str(FIGURES / "b_sweep_agreement.png"))


In [ ]:
Image(filename=str(FIGURES / "b_sweep_tradeoff.png"))


**Figures 5 and 6 — agreement and cost versus B.** This evidence was added
after the audit, at Lucas's request, as gate-2 presentation evidence — not a
new hypothesis, and not a retroactive edit to `design.md`'s declared-figure
list (`verification.md`, "Agreement and cost versus B"). It shows how the
analytical result relates to the same Monte-Carlo null it replaces, run two
ways on the identical S1 partition.

**Same-null convergence (Figure 5, `b_sweep_agreement.png`).** The
Monte-Carlo estimate here is drawn from the same capacity-stratified
partition the analytical adapter uses internally (transformed scores, raw
leave-target-out capacity, hurdle + adaptive bins, deficient-stratum merge)
— the only difference from the analytical value is exact moments versus a
finite number of resampling draws. Agreement with the analytical value rises
monotonically as `B` grows: rho = 0.9752 at `B = 20`, then 0.9922, 0.9979,
and 0.9995 at `B = 10,000` (mean over 3 seeds each, table above). Median
absolute z-error over the same span falls 0.198 -> 0.078 -> 0.026 -> 0.007.
Both series head toward the analytical row's own values (rho = 1, error =
0): the analytical result is this same null's `B -> infinity` limit,
computed exactly rather than approached by sampling.

The table's `median_seed_spread` column makes the same point from the noise
side: at `B = 20` the median spread between seeds is 0.401 z units, falling
to 0.133, 0.057, and 0.0196 as `B` grows, against the analytical row's
spread of exactly 0 (no seed dependence, by construction). At any given `B`,
the same-null MC estimate is a noisy version of the one answer the
analytical form returns exactly — not a different estimate converging on a
different target.

**Cost of closing that gap (Figure 6, `b_sweep_tradeoff.png`).** Matching
the analytical precision by raising `B` is expensive: the same-null MC
sweep at `B = 10,000` (3 seeds, 52 metapaths) took 173.08 s of wall-time
against the analytical closed form's 0.0064 s for the same 52 metapaths in
one query (`b-sweep-analytical-ref`, median of 5, no disk I/O) — a ratio of
roughly 27,000x, computed above directly from the table's own
`total_wall_time_s` cells. (`verification.md`'s own prose additionally
normalizes the 3-seed total to a per-seed basis — 173.08 s / 3 seeds ~=
57.7 s per seed — and reports that as "roughly 9,000x"; both figures
describe the same underlying measurement at a different unit of
comparison.) And even at that cost, `B = 10,000` does not reach the
analytical answer exactly (rho = 0.9995, not 1.0) — a residual finite-`B`
gap the table reports honestly.

**Separately: raising `b` was never the fix for the old, unstratified
null.** The old null (materialized from git history, the pre-this-task
implementation) at `b = 10,000` — 500x the app's own `b = 20` default — has
rho = 0.283 against the analytical value, barely different from its own
rho = 0.2605 at `b = 20` (Verify-step evidence: rank agreement, above). The
old null draws uniformly from the whole gene universe, ignoring
degree/capacity entirely; raising `b` there reduces its own sampling noise
around a degree-blind quantity, not the distance between that quantity and
the degree-stratified analytical answer. It is a different null, not a
noisier version of the same one, and no amount of `b` closes that gap.

**Caveats**, carried from `verification.md`: (a) the `old_null_b10k` row
uses a single seed (42), so no seed-to-seed spread is reported for it
(`n/a` in the table above), unlike the same-null MC rows, which average
over 3 seeds — the old-null point is not directly seed-range-comparable to
the same-null curve. (b) the same-null MC and analytical wall-times are
measured on the identical basis (cached S1 partitions, no disk I/O) and are
directly comparable; the old-null wall-time additionally includes matrix
disk I/O, so part of its 222.28 s is not "null generation" in the same
narrow sense, even though it is the correct number for what the real old
code path costs end-to-end.


## Behaviour changes

Two changes the design commits to (`design.md`, "Behaviour changes the
summary must present"), stated here as approved — not rewritten to match
the run — and then read against what was measured:

**1. Deterministic.** Identical queries return identical z. `b` and `seed`
are inert — they are still accepted as parameters (so no caller breaks), but
ignored, and passing either emits a `DeprecationWarning`. Two calls to
`query_metapath_z` with the same gene set and target return byte-identical
frames (`verification.md`, "Determinism (direct function, run1 vs run2
byte-identical frames): True"). **Measured: matches the promise.**

**2. Faster.** The design's promise, as approved and as restored by the
2026-09-02 reversal (`decisions.md`): "Per-metapath null cost falls from `b`
DWPC resamples to sub-millisecond moments (validated at 214x / 2,145x per
row against `B = 1,000` / `B = 10,000` Monte Carlo); the end-to-end query
number is this task's verify figure." **Measured: contradicts the promise
at the app's actual `b = 20` default** — the warm-matrix null step is
slower, not faster (70 ms vs 8.6 ms on `GiGiGpBP`), and end-to-end latency
is roughly unchanged rather than faster (103 s vs 85 s). See "Verify-step
evidence: timing" above for the full measurement; verdict there: hypothesis
NOT met. The 214x/2,145x figure is real, but it is validated against
`B = 1,000` / `B = 10,000` Monte Carlo draws, not against the `b = 20` the
live app actually uses.


## Conclusions and the reviewer's onward path

**The honest scorecard.** Each hypothesis quoted verbatim from `design.md`
as approved, against what this task measured:

- **Deterministic** ("identical queries return identical z; `b`/`seed` are
  inert") — **hypothesis met**: `new_df_1.equals(new_df_2) == True`
  (`verification.md`, "Positive control").
- **"Seed-to-seed MC spread is visible at default `b`; the analytical value
  has none."** — **hypothesis met**: `figures/mc_seed_spread.png` shows MC
  seed ranges up to roughly 19 z units at `b = 20` against an exact zero for
  the analytical value.
- **"Query latency drops by an order of magnitude or more."** —
  **hypothesis NOT met**: 103 s vs 85 s end-to-end, 70 ms vs 8.6 ms
  warm-matrix (Verify-step evidence: timing, above).
- **"The example query's analytical z correlates with the Monte-Carlo
  z."** — **hypothesis only weakly supported**: Spearman rho 0.2605
  (p = 0.0877, n = 44), not significant at alpha = 0.05 (Verify-step
  evidence: rank agreement, above).

`design.md` still states the latency and correlation hypotheses, and the
"Faster" behaviour change, in this original approved form — not rewritten
to match what was measured — because an earlier draft that did rewrite them
to agree with the measurement was reversed: `decisions.md` (2026-09-02)
rules that circular and establishes the standing rule that sections stating
pre-run expectations are never rewritten to match results. The divergence
between what the design predicted and what this notebook measured is itself
the finding this task reports, not a defect in either document.

- **Agreement versus B** (requested gate-2 evidence, presented as
  measurement, not a pre-run hypothesis): the same-null Monte Carlo
  converges monotonically to the analytical value as `B` grows (rho 0.9752
  -> 0.9922 -> 0.9979 -> 0.9995 for `B = 20 -> 10,000`) — the analytical
  result is that same null's `B -> infinity` limit, computed exactly.
  Matching its precision by raising `B` alone costs roughly 27,000x the
  analytical closed form's wall-time and still does not reach rho = 1.0.
  The old, unstratified null does not converge at all even at `b = 10,000`
  (rho 0.283) — a different, degree-blind null, not a noisier version of
  the analytical one (Verify-step evidence: agreement and cost versus B,
  above).

- [`audit.md`](audit.md) — the design read as 42 numbered claims against the
  landed tree, forward and reverse: **passes**. 18 documentary findings
  (F1-F18), 17 fixed and 1 recorded in `decisions.md` (the verify-step
  controls ran via direct `query_metapath_z` calls rather than through the
  live Streamlit app, for a documented memory reason); none required a code
  change.
- [`verification.md`](verification.md) — every command run and its real
  output, including the full `pytest -q` suite pass (65 passed, 4 subtests
  passed) and the memory-bounded methodology note that explains the
  end-to-end timing figure's disk-I/O dominance.
- The null's evidence base:
  [`docs/tasks/capacity-hurdle-adaptive-null/`](https://github.com/lagillenwater/multi-dwpc/tree/fix/random-null-stratified-srswor/docs/tasks/capacity-hurdle-adaptive-null)
  on branch `fix/random-null-stratified-srswor` (commit `3b0bdee`) — the
  calibration and pass-rate-history evidence `analytical_gene_set_z` is
  built on.

This is gate 2. Human review happens here, before the pull request opens.
